# Model 1: feature-group ablation 

**Question.** How much does each feature group contribute to ranking purchases, *inside a model* alongside the other features? Section 9.6 of `step2_model1_feature_engineering.ipynb` scored features one at a time; a feature can be weak alone and still help in combination, or strong alone and redundant in combination.

**Method.** A fixed **reference model** (gradient-boosted trees, LightGBM) is retrained on different feature sets; everything else (data, settings, seeds, number of trees) stays identical, so differences come from the features alone. The interim report describes Model 1 as a feature-based classifier (e.g. gradient-boosted trees) that outputs a purchase probability; the reference model follows that description but is **not** the final Model 1 (no tuning). Conclusions are about feature usefulness under this model, and the main ones should be re-checked once the final Model 1 is trained.

**Experiments** (all scored on val slates with a purchase, per-slate NDCG@10 as the headline metric):
1. **Objective check:** purchase-only classifier vs graded-weight classifier (vs LambdaRank as a reference) on all features. Picks the objective and the number of trees used everywhere else.
2. **Leave one group out:** all features minus one group at a time.
3. **Add one group at a time:** popularity, then item, brand, category history, then similarity.
4. **Finer checks:** recent-affinity features, and redundant pairs (raw vs log popularity, counts vs days-since).
5. **Core set:** brand and category history removed *together* (removing one at a time can hide redundancy between them), to test whether Model 1 can use a smaller feature set.

**Statistics.** Each main configuration is trained with 3 seeds and per-slate scores are averaged over seeds. Differences between configurations use a **paired bootstrap over slates** (both configurations rank the same slates, so shared slate difficulty cancels out). That interval covers *which slates were sampled*, not *training randomness*: two runs with different seeds differ a little even on identical features (`seed_sd` shows this). A difference counts as real only if its interval excludes 0 **and** it is clearly larger than the seed spread. Single-seed finer checks are the most exposed to this.

**Val, not test.** Every comparison here uses val; test is not touched. Val also sets the number of trees once (experiment 1), which makes all val scores slightly optimistic, but equally so for every configuration.

**Word2vec.** The comparison of word2vec against TF-IDF similarity is recorded in `appendix_w2v_experiment.ipynb` (section 9); this notebook uses the TF-IDF features only.

**Before running:** add a shortcut to the shared `Grp8Submission` folder in your My Drive.

**Inputs:** `features/model1/features_{train,val}.parquet` (from `step2_model1_feature_engineering.ipynb`; run it first).

**Outputs:** per-run results cached in `features/model1/ablation/` (reruns skip finished runs), summary tables printed below.

## 0. Setup

In [17]:
import os, time, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq
import lightgbm as lgb

# Project folders (same in every notebook)
from google.colab import drive
drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/Grp8Submission")
DATA = PROJECT / "data"
FEAT_M1 = PROJECT / "features" / "model1"
FEAT_M2 = PROJECT / "features" / "model2"
for d in (DATA, FEAT_M1, FEAT_M2):
    d.mkdir(parents=True, exist_ok=True)
FEAT = FEAT_M1
RESULTS = FEAT / "ablation"
RESULTS.mkdir(parents=True, exist_ok=True)

THREADS = os.cpu_count() or 4
SEEDS_MAIN = [0, 1, 2]          # reference, leave-one-out, add-one-in, core set
SEEDS_FINE = [0]                # finer checks (cheaper; compared against the reference's seed-0 run)
MAX_ROUNDS, EARLY_STOP = 2000, 100

BASE_PARAMS = dict(
    learning_rate=0.1, num_leaves=63, min_data_in_leaf=100,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=THREADS, verbose=-1, deterministic=True, force_row_wise=True,
)
GRADED_WEIGHT = {0: 1.0, 1: 1.0, 2: 5.0, 3: 10.0}   # click 1, cart 5, order 10 (interim report, weighted_action_score)
LAMBDARANK_GAIN = [0, 1, 3, 7]                        # 2**label - 1, as in the graded harness


def load(path, columns=None):
    # ignore_metadata avoids the pandas 2.x/3.x pyarrow-metadata TypeError
    return pq.read_table(path, columns=columns).to_pandas(ignore_metadata=True)

print("threads:", THREADS, "| lightgbm", lgb.__version__)

threads: 15 | lightgbm 4.7.0


## 1. Feature groups
The same grouping as the single-feature analysis. `current_query_length` and `candidate_has_metadata` are **context** and kept in every configuration: neither ranks on its own (both score exactly at random in 9.6), but they tell the model how much to trust other features (e.g. why brand, category and similarity are missing).

In [18]:
GROUPS = {
    "popularity": ["candidate_popularity_count", "candidate_log_popularity",
                   "candidate_log_pop_cart", "candidate_log_pop_ord"],
    "item_history": ["item_click_cnt_7d", "item_cart_cnt_90d", "item_ord_cnt_365d",
                     "item_days_since_cart", "item_days_since_ord"],
    "brand_history": ["recent_brand_affinity_1d", "recent_brand_affinity_3d", "recent_brand_affinity_7d",
                      "brand_click_cnt_7d", "brand_cart_cnt_90d", "brand_ord_cnt_365d",
                      "brand_days_since_cart", "brand_days_since_ord"],
    "category_history": ["recent_cate_affinity_1d", "recent_cate_affinity_3d", "recent_cate_affinity_7d",
                         "cate_click_cnt_7d", "cate_cart_cnt_90d", "cate_ord_cnt_365d"],
    "similarity_tfidf": ["query_candidate_name_sim", "query_candidate_cate4_sim"],
}
CONTEXT = ["current_query_length", "candidate_has_metadata"]
RECENT = [f"recent_{k}_affinity_{d}d" for k in ("brand", "cate") for d in (1, 3, 7)]
ALL = CONTEXT + [f for g in GROUPS.values() for f in g]          # reference: the existing 27 features (TF-IDF)
print({g: len(f) for g, f in GROUPS.items()}, "| context", len(CONTEXT), "| reference total", len(ALL))

{'popularity': 4, 'item_history': 5, 'brand_history': 8, 'category_history': 6, 'similarity_tfidf': 2} | context 2 | reference total 27


## 2. Data
Train rows are sorted by slate (LambdaRank and LightGBM's NDCG metric need each slate's rows together). Only val slates with a purchase are scored, which is all the purchase metrics use.

In [19]:
def load_split(split):
    f = load(FEAT / f"features_{split}.parquet")
    missing = [c for c in ALL if c not in f.columns]
    assert not missing, f"missing columns: {missing}"
    return f.sort_values("slate_id", kind="stable").reset_index(drop=True)


tr = load_split("train")
va = load_split("val")
va = va[va.groupby("slate_id")["label"].transform(lambda s: (s == 3).any())].reset_index(drop=True)
tr_groups = tr.groupby("slate_id", sort=False).size().to_numpy()
va_groups = va.groupby("slate_id", sort=False).size().to_numpy()
print(f"train: {len(tr):,} rows, {len(tr_groups):,} slates, purchase rate {tr['label'].eq(3).mean():.4%}")
print(f"val (purchase slates): {len(va):,} rows, {len(va_groups):,} slates")

train: 2,988,641 rows, 139,882 slates, purchase rate 0.5618%
val (purchase slates): 184,165 rows, 1,930 slates


## 3. Evaluation helpers
Per-slate metrics: identical code to section 9.6 of the feature notebook. `paired_diff` bootstraps the mean per-slate difference between two configurations over slates.

In [20]:
METRICS = ["ndcg@10", "hit@10", "hit@1", "mrr"]


def _gains(label, relevance):
    label = np.asarray(label)
    if relevance == "purchase":
        return (label == 3).astype(np.float64)
    if relevance == "graded":
        return (2.0 ** label - 1.0).astype(np.float64)
    raise ValueError("relevance must be 'purchase' or 'graded'")


def _group_layout(sorted_slates):
    n = len(sorted_slates)
    start = np.r_[0, np.flatnonzero(sorted_slates[1:] != sorted_slates[:-1]) + 1]
    size = np.diff(np.r_[start, n])
    rank = np.arange(n) - np.repeat(start, size) + 1
    return start, rank


def per_slate_metrics(df, score_col, k=10, relevance="purchase", higher_is_better=True,
                      n_tiebreaks=5, seed=0):
    slate = df["slate_id"].to_numpy()
    gain = _gains(df["label"].to_numpy(), relevance)
    has_rel = pd.Series(gain > 0).groupby(slate).transform("any").to_numpy()
    slate, gain = slate[has_rel], gain[has_rel]
    score = df[score_col].to_numpy(dtype=np.float64)[has_rel]
    if len(slate) == 0:
        raise ValueError("no slates with a relevant item")
    key = score if higher_is_better else -score
    key = np.where(np.isnan(key), -np.inf, key)
    ideal_order = np.lexsort((-gain, slate))
    start, rank = _group_layout(slate[ideal_order])
    disc = np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0)
    idcg = np.add.reduceat(gain[ideal_order] * disc, start)
    rng = np.random.default_rng(seed)
    acc = {m: np.zeros(len(start)) for m in METRICS}
    for _ in range(n_tiebreaks):
        tie = rng.random(len(slate))
        order = np.lexsort((tie, -key, slate))
        g = gain[order]
        rel = g > 0
        acc["ndcg@10"] += np.add.reduceat(g * disc, start) / idcg
        acc["hit@10"] += np.add.reduceat((rel & (rank <= k)).astype(float), start) > 0
        acc["hit@1"] += np.add.reduceat((rel & (rank == 1)).astype(float), start) > 0
        first = np.minimum.reduceat(np.where(rel, rank, np.inf), start)
        acc["mrr"] += 1.0 / first
    out = pd.DataFrame({m: v / n_tiebreaks for m, v in acc.items()})
    out = out.rename(columns={"ndcg@10": f"ndcg@{k}", "hit@10": f"hit@{k}"})
    out.insert(0, "slate_id", slate[ideal_order][start])
    return out


def paired_diff(a, b, col="ndcg@10", n_boot=2000, seed=0):
    # mean of a - b over slates, with a 95% bootstrap interval (a and b: per-slate tables on the same slates)
    assert (a["slate_id"].to_numpy() == b["slate_id"].to_numpy()).all(), "slates not aligned"
    d = a[col].to_numpy() - b[col].to_numpy()
    rng = np.random.default_rng(seed)
    boot = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"diff": d.mean(), "ci_low": lo, "ci_high": hi}

## 4. Training a configuration
- **`purchase`**: binary classifier, target = purchase. Outputs a purchase probability directly.
- **`graded`**: binary classifier, target = any interaction, with sample weights click 1 / cart 5 / purchase 10 (interim report §2.2.5: clicks and carts as weaker signals through graded weights).
- **`lambdarank`**: LightGBM's ranking objective with graded relevance. Reference only: it outputs ranking scores, not probabilities, so it does not fit the report's design of blending P1 and P2.

Each finished run's per-slate scores are saved under a key built from the feature set, objective, seed and number of trees, so reruns load them instead of retraining. A configuration that repeats an earlier feature set (e.g. the reference appears in several experiments) is trained once.

In [21]:
def _dataset(df, feats, objective, groups):
    lab = df["label"].to_numpy()
    if objective == "purchase":
        return lgb.Dataset(df[feats], label=(lab == 3).astype(int), group=groups, free_raw_data=True)
    if objective == "graded":
        return lgb.Dataset(df[feats], label=(lab >= 1).astype(int), group=groups,
                           weight=pd.Series(lab).map(GRADED_WEIGHT).to_numpy(), free_raw_data=True)
    if objective == "lambdarank":
        return lgb.Dataset(df[feats], label=lab, group=groups, free_raw_data=True)
    raise ValueError(objective)


def _params(objective, seed):
    p = dict(BASE_PARAMS, seed=seed, bagging_seed=seed, feature_fraction_seed=seed, data_random_seed=seed)
    if objective == "lambdarank":
        p.update(objective="lambdarank", label_gain=LAMBDARANK_GAIN)
    else:
        p.update(objective="binary")
    return p


def fit(feats, objective, seed, n_rounds=None):
    # n_rounds None -> early stopping on val NDCG@10 (purchase relevance); otherwise a fixed number of trees
    dtrain = _dataset(tr, feats, objective, tr_groups)
    params = _params(objective, seed)
    if n_rounds is None:
        dval = lgb.Dataset(va[feats], label=va["label"].eq(3).astype(int).to_numpy(), group=va_groups, reference=dtrain)
        params.update(metric="ndcg", eval_at=[10])
        model = lgb.train(params, dtrain, MAX_ROUNDS, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    else:
        params.update(metric="None")
        model = lgb.train(params, dtrain, n_rounds)
    return model


def score(model, feats):
    pred = model.predict(va[feats], num_iteration=model.best_iteration or None, num_threads=THREADS)
    return per_slate_metrics(va.assign(_score=pred), "_score")


def run_key(feats, objective, seed, n_rounds):
    blob = json.dumps({"f": sorted(feats), "o": objective, "s": seed, "n": n_rounds, "p": BASE_PARAMS,
                       "w": GRADED_WEIGHT}, sort_keys=True, default=str)
    return hashlib.md5(blob.encode()).hexdigest()[:16]


def run(feats, objective, seed, n_rounds):
    # per-slate metrics for one (feature set, objective, seed); cached on disk
    path = RESULTS / f"{objective}_{run_key(feats, objective, seed, n_rounds)}.parquet"
    if path.exists():
        return load(path)
    t0 = time.time()
    model = fit(feats, objective, seed, n_rounds)
    ps = score(model, feats)
    ps.to_parquet(path, index=False)
    print(f"  trained {len(feats)} features, {objective}, seed {seed}: {time.time() - t0:.0f}s")
    return ps

## 5. Experiment 1: objective check
All features (TF-IDF similarity), seed 0, early stopping on val. The classifier objective with the higher NDCG@10 is used for the rest of the notebook, with its best number of trees fixed for every configuration. Fixing the trees means no configuration gets its own early stopping on val, which keeps comparisons fair. LambdaRank is shown for reference: if it is far ahead, that is worth raising when choosing the final Model 1.

In [22]:
obj_rows, obj_rounds = {}, {}
for objective in ["purchase", "graded", "lambdarank"]:
    t0 = time.time()
    model = fit(ALL, objective, seed=0)
    ps = score(model, ALL)
    obj_rounds[objective] = model.best_iteration
    obj_rows[objective] = {**ps[METRICS].mean().to_dict(), "best_trees": model.best_iteration,
                           "minutes": (time.time() - t0) / 60}
    print(f"{objective}: {obj_rows[objective]['ndcg@10']:.4f} NDCG@10, {model.best_iteration} trees, "
          f"{obj_rows[objective]['minutes']:.1f} min")
objectives = pd.DataFrame(obj_rows).T
print(objectives.round(4).to_string())

purchase: 0.3553 NDCG@10, 40 trees, 0.1 min
graded: 0.3808 NDCG@10, 173 trees, 0.1 min
lambdarank: 0.3843 NDCG@10, 38 trees, 0.1 min
            ndcg@10  hit@10   hit@1     mrr  best_trees  minutes
purchase     0.3553  0.5939  0.1705  0.3012        40.0   0.0791
graded       0.3808  0.6087  0.2002  0.3297       173.0   0.1384
lambdarank   0.3843  0.6123  0.2039  0.3327        38.0   0.0929


In [23]:
OBJECTIVE = max(["purchase", "graded"], key=lambda o: objectives.loc[o, "ndcg@10"])   # override by hand if needed
N_ROUNDS = int(obj_rounds[OBJECTIVE])
print("objective for the ablation:", OBJECTIVE, "| trees:", N_ROUNDS)

objective for the ablation: graded | trees: 173


## 6. Experiments 2–5: configurations
Every configuration is a feature set. Identical sets across experiments are trained once (e.g. "core without similarity" equals the "popularity + item_history" step).

In [ ]:
configs = []   # (experiment, name, features, seeds)
def add(exp, name, feats, seeds=SEEDS_MAIN):
    configs.append((exp, name, list(dict.fromkeys(feats)), seeds))

add("reference", "all features (TF-IDF)", ALL)

for g in GROUPS:                                                   # 2. leave one group out
    add("leave_one_out", f"without {g}", [f for f in ALL if f not in GROUPS[g]])

order = ["popularity", "item_history", "brand_history", "category_history", "similarity_tfidf"]
for i in range(1, len(order) + 1):                                 # 3. add one group at a time
    add("add_one_in", " + ".join(order[:i]), CONTEXT + [f for g in order[:i] for f in GROUPS[g]])

drop = lambda cols: [f for f in ALL if f not in cols]               # 4. finer checks (one seed)
add("finer", "without all recent affinity", drop(RECENT), SEEDS_FINE)
add("finer", "without recent brand affinity", drop(RECENT[:3]), SEEDS_FINE)
add("finer", "without recent category affinity", drop(RECENT[3:]), SEEDS_FINE)
add("finer", "without raw popularity count", drop(["candidate_popularity_count"]), SEEDS_FINE)
add("finer", "item: counts only (no days-since)", drop(["item_days_since_cart", "item_days_since_ord"]), SEEDS_FINE)
add("finer", "item: days-since only (no cart/order counts)", drop(["item_cart_cnt_90d", "item_ord_cnt_365d"]), SEEDS_FINE)
add("finer", "brand: counts only (no days-since)", drop(["brand_days_since_cart", "brand_days_since_ord"]), SEEDS_FINE)

CORE = CONTEXT + GROUPS["popularity"] + GROUPS["item_history"] + GROUPS["similarity_tfidf"]   # 5. core set
CORE_NO_SIM = [f for f in CORE if f not in GROUPS["similarity_tfidf"]]
add("core", "core: popularity + item + similarity", CORE)
add("core", "core without similarity", CORE_NO_SIM)
add("core", "core, log popularity only", [f for f in CORE if f != "candidate_popularity_count"])

jobs = {(tuple(sorted(f)), s) for _, _, f, seeds in configs for s in seeds}
print(f"{len(configs)} configurations -> {len(jobs)} distinct training runs "
      f"(objective-check models not counted; cached runs are skipped)")

In [26]:
t_all = time.time()
per_slate = {}   # (sorted features, seed) -> per-slate metrics
for exp, name, feats, seeds in configs:
    for s in seeds:
        key = (tuple(sorted(feats)), s)
        if key not in per_slate:
            per_slate[key] = run(feats, OBJECTIVE, s, N_ROUNDS)
print(f"all runs done in {(time.time() - t_all) / 60:.1f} min")

  trained 13 features, graded, seed 0: 5s
  trained 13 features, graded, seed 1: 5s
  trained 13 features, graded, seed 2: 5s
  trained 12 features, graded, seed 0: 5s
  trained 12 features, graded, seed 1: 5s
  trained 12 features, graded, seed 2: 5s
all runs done in 0.5 min


## 7. Results
For each configuration: mean NDCG@10 and HitRate@10 (per-slate scores averaged over its seeds), `seed_sd` (spread of the per-seed means: the size of pure training noise), and the paired difference from a baseline with its 95% interval. `ci_excludes_0` marks differences whose interval excludes 0; read it together with `seed_sd` before calling a difference real. Configurations with one seed are compared against the baseline's seed-0 run, so both sides use the same seed.

In [27]:
def seed_avg(feats, seeds):
    tabs = [per_slate[(tuple(sorted(feats)), s)] for s in seeds]
    out = tabs[0][["slate_id"]].copy()
    for m in METRICS:
        out[m] = np.mean([t[m].to_numpy() for t in tabs], axis=0)
    return out


def results_table(exp, baseline=None, sequential=False):
    rows, prev = [], None
    for e, name, feats, seeds in configs:
        if e != exp:
            continue
        ps = seed_avg(feats, seeds)
        per_seed = [per_slate[(tuple(sorted(feats)), s)]["ndcg@10"].mean() for s in seeds]
        row = {"configuration": name, "n_features": len(feats), "seeds": len(seeds),
               "ndcg@10": ps["ndcg@10"].mean(), "seed_sd": np.std(per_seed) if len(per_seed) > 1 else np.nan,
               "hit@10": ps["hit@10"].mean()}
        base = prev if sequential else baseline
        if base is not None:
            common = [s for s in seeds if s in base[1]]
            r = paired_diff(ps if common == seeds else seed_avg(feats, common), seed_avg(base[0], common))
            row.update({"diff": r["diff"], "ci_low": r["ci_low"], "ci_high": r["ci_high"],
                        "ci_excludes_0": "yes" if r["ci_low"] > 0 or r["ci_high"] < 0 else ""})
        rows.append(row)
        prev = (feats, seeds)
    return pd.DataFrame(rows).set_index("configuration")


REF = (ALL, SEEDS_MAIN)
pd.set_option("display.width", 200)

### 7.1 Leave one group out (difference = configuration − all features)
A large negative difference means the group carries information nothing else provides; about 0 means it is redundant given the others; positive means it hurts.

In [28]:
print(results_table("reference").round(4).to_string())
print()
print(results_table("leave_one_out", baseline=REF).round(4).to_string())

                       n_features  seeds  ndcg@10  seed_sd  hit@10
configuration                                                     
all features (TF-IDF)          27      3   0.3799   0.0012  0.6082

                          n_features  seeds  ndcg@10  seed_sd  hit@10    diff  ci_low  ci_high ci_excludes_0
configuration                                                                                               
without popularity                23      3   0.1517   0.0018  0.2461 -0.2282 -0.2439  -0.2127           yes
without item_history              22      3   0.3605   0.0018  0.5991 -0.0193 -0.0267  -0.0123           yes
without brand_history             19      3   0.3774   0.0009  0.6054 -0.0024 -0.0054   0.0008              
without category_history          21      3   0.3774   0.0011  0.6035 -0.0025 -0.0056   0.0007              
without similarity_tfidf          25      3   0.3749   0.0019  0.6012 -0.0050 -0.0110   0.0014              


### 7.2 Add one group at a time (difference = this step − previous step)

In [29]:
print(results_table("add_one_in", sequential=True).round(4).to_string())

                                                                                 n_features  seeds  ndcg@10  seed_sd  hit@10    diff  ci_low  ci_high ci_excludes_0
configuration                                                                                                                                                      
popularity                                                                                6      3   0.3402   0.0005  0.5805     NaN     NaN      NaN           NaN
popularity + item_history                                                                11      3   0.3716   0.0003  0.6040  0.0314  0.0238   0.0392           yes
popularity + item_history + brand_history                                                19      3   0.3739   0.0007  0.6023  0.0023 -0.0021   0.0066              
popularity + item_history + brand_history + category_history                             25      3   0.3749   0.0019  0.6012  0.0010 -0.0021   0.0040              
popularity + ite

### 7.3 Finer checks (one seed; difference = configuration − all features, seed 0)
One seed, so training noise is not averaged out. As a yardstick, use the reference's `seed_sd` from 7.1 and the raw-popularity row: removing the raw count loses no information (the log version is kept and trees are unaffected by a log), so whatever difference it shows is noise. Differences of similar size elsewhere should not be read as real; rerun those checks with `SEEDS_MAIN` if one matters for a decision.

In [31]:
print(results_table("finer", baseline=REF).round(4).to_string())

                                              n_features  seeds  ndcg@10  seed_sd  hit@10    diff  ci_low  ci_high ci_excludes_0
configuration                                                                                                                   
without all recent affinity                           21      1   0.3797      NaN  0.6075 -0.0011 -0.0057   0.0035              
without recent brand affinity                         24      1   0.3761      NaN  0.6057 -0.0046 -0.0089  -0.0002           yes
without recent category affinity                      24      1   0.3788      NaN  0.6057 -0.0020 -0.0068   0.0026              
without raw popularity count                          26      1   0.3766      NaN  0.6062 -0.0041 -0.0087   0.0004              
item: counts only (no days-since)                     25      1   0.3825      NaN  0.6111  0.0018 -0.0029   0.0065              
item: days-since only (no cart/order counts)          25      1   0.3783      NaN  0.6039 -0.0024

### 7.4 Core feature set (difference = configuration − all 27 features)
Leave-one-out removed brand and category history one at a time; each cost about 0.002, within noise. Here both are removed **together**. If the core set matches all 27 features (interval includes 0, difference small next to `seed_sd`), Model 1 can use the smaller set, which also fits the design: long-term brand and category preferences are Model 2's job. The last line tests similarity *within* the core set, where it may show more clearly than in the full model. "Log popularity only" is a tidy-up check: it should be within noise, confirming the raw count can go.

In [32]:
print(results_table("core", baseline=REF).round(4).to_string())
r = paired_diff(seed_avg(CORE, SEEDS_MAIN), seed_avg(CORE_NO_SIM, SEEDS_MAIN))
print(f"\nsimilarity within the core set (core - core without similarity): "
      f"{r['diff']:+.4f} (95% CI {r['ci_low']:+.4f} to {r['ci_high']:+.4f})")

                                      n_features  seeds  ndcg@10  seed_sd  hit@10    diff  ci_low  ci_high ci_excludes_0
configuration                                                                                                           
core: popularity + item + similarity          13      3   0.3737   0.0008  0.6016 -0.0061 -0.0100  -0.0025           yes
core without similarity                       11      3   0.3716   0.0003  0.6040 -0.0082 -0.0152  -0.0008           yes
core, log popularity only                     12      3   0.3760   0.0012  0.6060 -0.0039 -0.0076  -0.0001           yes

similarity within the core set (core - core without similarity): +0.0021 (95% CI -0.0048 to +0.0086)


### 7.5 What the reference model relies on
LightGBM's gain importance for the all-features model (seed 0), as a third view next to single-feature scores (9.6 of the feature notebook) and the ablation. Importance shows what the model *uses*; the ablation shows what it would *lose*. A feature can have high importance and still be removable if a correlated feature takes over (e.g. raw and log popularity split the credit for the same information).

In [33]:
model = fit(ALL, OBJECTIVE, seed=0, n_rounds=N_ROUNDS)
imp = pd.Series(model.feature_importance("gain"), index=ALL)
imp = (imp / imp.sum() * 100).sort_values(ascending=False)
group_of = {f: g for g, fs in GROUPS.items() for f in fs} | {f: "context" for f in CONTEXT}
print(pd.DataFrame({"gain_%": imp.round(2), "group": [group_of[f] for f in imp.index]}).to_string())
print("\nby group (%):", imp.groupby(lambda f: group_of[f]).sum().round(1).sort_values(ascending=False).to_dict())

                            gain_%             group
candidate_popularity_count   54.24        popularity
candidate_log_popularity     15.35        popularity
query_candidate_name_sim      6.73  similarity_tfidf
item_click_cnt_7d             3.24      item_history
candidate_log_pop_ord         3.17        popularity
query_candidate_cate4_sim     2.87  similarity_tfidf
item_cart_cnt_90d             2.34      item_history
current_query_length          1.90           context
item_days_since_cart          1.00      item_history
brand_days_since_cart         0.93     brand_history
brand_days_since_ord          0.87     brand_history
recent_cate_affinity_7d       0.78  category_history
cate_cart_cnt_90d             0.77  category_history
item_days_since_ord           0.75      item_history
candidate_log_pop_cart        0.70        popularity
cate_click_cnt_7d             0.61  category_history
item_ord_cnt_365d             0.58      item_history
recent_cate_affinity_3d       0.50  category_h

## 8. Next steps
1. Agree the final Model 1 feature set from 7.1–7.4: drop groups or features whose removal does not hurt (interval includes or sits above 0), keep the rest.
2. Once the final Model 1 is chosen and trained, re-check the main conclusions there by retraining it without the groups this ablation flagged.
3. The objective check (section 5) is evidence for the final Model 1's training objective; P1 will still need calibration on val before blending with P2, because training on negative-sampled slates inflates predicted probabilities.